# Task 9 - Relationships Between Metrics

## Analytical Question

**What relationships were observed between Ethereum network activity metrics during the analysis period?**

The analysis investigates:

- transactions vs active addresses;
- transactions vs gas used;
- gas price vs network activity;
- active addresses vs transactions per active address;
- gas utilization vs transaction activity.

### Workflow

**Existing analytical datasets → Python/Pandas → correlation analysis → scatter plots → trend analysis → insight**

### Analysis window

- Full timestamp window: **2026-08-24 10:01:35 UTC → 2026-09-23 10:01:35 UTC**
- Partial days excluded: **August 24 and September 23, 2026**
- Comparable analysis period: **29 complete days, August 25 → September 22, 2026**

> Correlation describes statistical association. It does not by itself establish causation.


## 1. Load the Analytical Datasets

Task 9 reuses the daily analytical outputs produced by the earlier tasks rather than loading raw blockchain-scale data into Pandas.


In [127]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

transactions = pd.read_csv("../data/processed/daily_transactions.csv")
active_addresses = pd.read_csv("../data/processed/daily_active_addresses.csv")
network_utilization = pd.read_csv("../data/processed/daily_network_utilization.csv")
gas_activity = pd.read_csv("../data/processed/daily_gas_activity.csv")


## 2. Prepare the Analysis Dataset

Convert the date fields to datetime, combine the daily analytical results, calculate transactions per active address, and exclude the two partial boundary days to ensure that the daily observations are comparable.

In [134]:
# Convert all dates to proper dates for analysis
for df in [transactions, active_addresses, network_utilization, gas_activity]:
    df["date"] = pd.to_datetime(df["date"])

# The two partial days
start_date = df["date"].min()
end_date = df["date"].max()

analysis_df = (
    transactions
    .merge(
        active_addresses,
        on="date",
        how="inner"
    )
    .merge(
        network_utilization,
        on="date",
        how="inner"
    )
    .merge(
        gas_activity[
            [
                "date",
                "avg_gas_price_gwei",
                "median_gas_price_gwei"
            ]
        ],
        on="date",
        how="inner"
    )
)

analysis_df["transactions_per_active_address"] = (
    analysis_df["transaction_count"]
    / analysis_df["active_addresses"]
)


analysis_df = (
    analysis_df[
        (analysis_df["date"] > start_date)
        & (analysis_df["date"] < end_date)
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

analysis_df.head()

,date,transaction_count,active_addresses,avg_transactions_per_block,total_gas_used,avg_gas_used_per_block,avg_gas_limit,avg_gas_utilization_pct,avg_gas_price_gwei,median_gas_price_gwei,transactions_per_active_address
0,2026-08-25,1994723,672803,278.048927,218050922535,3.039461e+07,5.999670e+07,50.660459,0.846301,0.282506,2.964795
1,2026-08-26,1898276,612723,264.678751,217769296656,3.036382e+07,5.999638e+07,50.609431,0.766541,0.203725,3.098098
2,2026-08-27,1931985,645948,269.078691,218437823137,3.042310e+07,5.999644e+07,50.708048,0.916385,0.298498,2.990930
3,2026-08-28,1902964,616346,265.332404,218212568813,3.042562e+07,5.999674e+07,50.712133,0.886904,0.301275,3.087493
4,2026-08-29,2034340,578578,283.452696,218478110496,3.044143e+07,5.999613e+07,50.738994,0.464875,0.063080,3.516103


## 3. Validate the Analytical Dataset

Expected result:

- **29 rows** — one for each complete day;
- one row per date;
- no duplicate dates;
- no missing values in the metrics used for correlation analysis.


In [135]:
print("Shape:", analysis_df.shape)
print("Date range:", analysis_df["date"].min(), "to", analysis_df["date"].max())
print("Duplicate dates:", analysis_df["date"].duplicated().sum())

print("\nMissing values:")
print(analysis_df.isna().sum())

analysis_df.head()


Shape: (29, 11)
Date range: 2026-08-25 00:00:00 to 2026-09-22 00:00:00
Duplicate dates: 0

Missing values:
date                               0
transaction_count                  0
active_addresses                   0
avg_transactions_per_block         0
total_gas_used                     0
avg_gas_used_per_block             0
avg_gas_limit                      0
avg_gas_utilization_pct            0
avg_gas_price_gwei                 0
median_gas_price_gwei              0
transactions_per_active_address    0
dtype: int64


,date,transaction_count,active_addresses,avg_transactions_per_block,total_gas_used,avg_gas_used_per_block,avg_gas_limit,avg_gas_utilization_pct,avg_gas_price_gwei,median_gas_price_gwei,transactions_per_active_address
0,2026-08-25,1994723,672803,278.048927,218050922535,3.039461e+07,5.999670e+07,50.660459,0.846301,0.282506,2.964795
1,2026-08-26,1898276,612723,264.678751,217769296656,3.036382e+07,5.999638e+07,50.609431,0.766541,0.203725,3.098098
2,2026-08-27,1931985,645948,269.078691,218437823137,3.042310e+07,5.999644e+07,50.708048,0.916385,0.298498,2.990930
3,2026-08-28,1902964,616346,265.332404,218212568813,3.042562e+07,5.999674e+07,50.712133,0.886904,0.301275,3.087493
4,2026-08-29,2034340,578578,283.452696,218478110496,3.044143e+07,5.999613e+07,50.738994,0.464875,0.063080,3.516103


## 4. Select Metrics for Relationship Analysis

The following variables are used:

| Metric | Description |
|---|---|
| `transaction_count` | Daily Ethereum transaction count |
| `active_addresses` | Daily unique addresses appearing as sender or recipient |
| `transactions_per_active_address` | Daily transactions divided by daily active addresses |
| `avg_transactions_per_block` | Average transactions included per block |
| `total_gas_used` | Total gas consumed across the day's blocks |
| `avg_gas_price_gwei` | Daily average transaction gas price |
| `avg_gas_utilization_pct` | Average daily block gas utilization |


## 5. Correlation Analysis

Pearson correlation is used to measure the **strength and direction of linear relationships** between the selected Ethereum network activity metrics across the daily observations.

Interpretation:

- **closer to +1** → strong positive linear association;
- **closer to 0** → weak or no linear association;
- **closer to −1** → strong negative linear association.

A correlation close to zero does not necessarily mean that no relationship exists; it indicates that there is little or no **linear** association between the variables.

**Correlation does not establish causation.**

In [112]:
relationship_metrics = [
    "transaction_count",
    "active_addresses",
    "transactions_per_active_address",
    "avg_transactions_per_block",
    "total_gas_used",
    "avg_gas_price_gwei",
    "avg_gas_utilization_pct",
]

correlation_matrix = analysis_df[relationship_metrics].corr()

correlation_matrix.round(3)


,transaction_count,active_addresses,transactions_per_active_address,avg_transactions_per_block,total_gas_used,avg_gas_price_gwei,avg_gas_utilization_pct
transaction_count,1.000,0.874,-0.483,1.000,-0.223,0.065,-0.115
active_addresses,0.874,1.000,-0.839,0.874,-0.276,0.136,-0.226
transactions_per_active_address,-0.483,-0.839,1.000,-0.483,0.286,-0.206,0.321
avg_transactions_per_block,1.000,0.874,-0.483,1.000,-0.234,0.070,-0.122
total_gas_used,-0.223,-0.276,0.286,-0.234,1.000,-0.390,0.910
avg_gas_price_gwei,0.065,0.136,-0.206,0.070,-0.390,1.000,-0.365
avg_gas_utilization_pct,-0.115,-0.226,0.321,-0.122,0.910,-0.365,1.000


## 6. Relationship Analysis

The analysis examines five relationships between Ethereum network activity metrics to assess the strength and direction of their linear associations.

In [ ]:
relationships = {
    "Transactions vs active addresses": (
        "transaction_count",
        "active_addresses",
    ),
    "Transactions vs gas used": (
        "transaction_count",
        "total_gas_used",
    ),
    "Gas price vs network activity": (
        "avg_gas_price_gwei",
        "transaction_count",
    ),
    "Active addresses vs transactions per active address": (
        "active_addresses",
        "transactions_per_active_address",
    ),
    "Gas utilization vs transaction activity": (
        "avg_gas_utilization_pct",
        "transaction_count",
    ),
}

relationship_results = []

for name, (x, y) in relationships.items():
    r = analysis_df[x].corr(analysis_df[y])
    relationship_results.append({
        "relationship": name,
        "correlation": r,
        "absolute_correlation": abs(r),
    })

relationship_results = pd.DataFrame(relationship_results)
relationship_results = relationship_results.sort_values(
    "absolute_correlation",
    ascending=False
).reset_index(drop=True)

relationship_results[["relationship", "correlation"]].round(3)


## 7. Visual Analysis of Relationships

The scatter plots provide visual context for the correlation results and allow the observed relationships to be examined across the daily observations.

In [ ]:
# Matplotlib visuals

# plot_pairs = [
#     ("transaction_count", "active_addresses", "Transactions vs Active Addresses"),
#     ("transaction_count", "total_gas_used", "Transactions vs Total Gas Used"),
#     ("avg_gas_price_gwei", "transaction_count", "Gas Price vs Transactions"),
#     (
#         "active_addresses",
#         "transactions_per_active_address",
#         "Active Addresses vs Transactions per Active Address",
#     ),
#     (
#         "avg_gas_utilization_pct",
#         "transaction_count",
#         "Gas Utilization vs Transactions",
#     ),
# ]

# for x, y, title in plot_pairs:
#     plt.figure(figsize=(8, 5))
#     plt.scatter(analysis_df[x], analysis_df[y], alpha=0.7)
#     plt.title(title)
#     plt.xlabel(x.replace("_", " ").title())
#     plt.ylabel(y.replace("_", " ").title())
#     plt.grid(alpha=0.2)
#     plt.tight_layout()
#     plt.show()


## Plotly Visuals

## 1. Transactions Vs Active Addresses 

In [ ]:
fig = px.scatter(
    analysis_df,
    x="active_addresses",
    y="transaction_count",
    title="Transactions vs Active Addresses (r = 0.874)",
    labels={
        "active_addresses": "Active Addresses",
        "transaction_count": "Transaction Count"
    },
    trendline="ols",
    custom_data=["date"]
)

fig.update_layout(
    template="plotly_dark",
    hovermode="closest",
    hoverlabel=dict(
        bgcolor="rgba(30, 30, 30, 0.80)",
        font_size=13
    ),
    height=600,
    margin=dict(l=70, r=40, t=100, b=80),
    xaxis=dict(
        title="Active Addresses",
        fixedrange=False
    ),
    yaxis=dict(
        title="Transaction Count",
        tickformat=",",
        fixedrange=True
    )
)

fig.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "<b>%{customdata[0]|%b %d, %Y}</b><br>"
        "Active Addresses: %{x:,.0f}<br>"
        "Transactions: %{y:,.0f}"
        "<extra></extra>"
    )
)

# Trendline color
fig.data[-1].update(
    line=dict(color="orange", width=1)
)

fig.show()

## 2. Active Addresses vs Transactions per Active Address

In [ ]:
fig = px.scatter(
    analysis_df,
    x="active_addresses",
    y="transactions_per_active_address",
    title="Active Addresses vs Transactions per Active Address (r = -0.839)",
    labels={
        "active_addresses": "Active Addresses",
        "transactions_per_active_address": "Transactions per Active Address"
    },
    trendline="ols",
    custom_data=["date"]
)

fig.update_layout(
    template="plotly_dark",
    hovermode="closest",
    hoverlabel=dict(
        bgcolor="rgba(30, 30, 30, 0.80)",
        font_size=13
    ),
    height=600,
    margin=dict(l=70, r=40, t=100, b=80),
    xaxis=dict(
        title="Active Addresses",
        fixedrange=False
    ),
    yaxis=dict(
        title="Transactions per Active Address",
        tickformat=".2f",
        fixedrange=True
    )
)

fig.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "<b>%{customdata[0]|%b %d, %Y}</b><br>"
        "Active Addresses: %{x:,.0f}<br>"
        "Transactions per Active Address: %{y:.2f}"
        "<extra></extra>"
    )
)

# Trendline color
fig.data[-1].update(
    line=dict(color="orange", width=1)
)

fig.show()

## 3. Transactions vs Gas Used

In [ ]:
fig = px.scatter(
    analysis_df,
    x="total_gas_used",
    y="transaction_count",
    title="Transactions vs Gas Used (r = -0.223)",
    labels={
        "total_gas_used": "Total Gas Used",
        "transaction_count": "Transaction Count"
    },
    trendline="ols",
    custom_data=["date"]
)

fig.update_layout(
    template="plotly_dark",
    hovermode="closest",
    hoverlabel=dict(
        bgcolor="rgba(30, 30, 30, 0.80)",
        font_size=13
    ),
    height=600,
    margin=dict(l=70, r=40, t=100, b=80),
    xaxis=dict(
        title="Total Gas Used",
        tickformat="~s",
        fixedrange=False
    ),
    yaxis=dict(
        title="Transaction Count",
        tickformat=",",
        fixedrange=True
    )
)

fig.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "<b>%{customdata[0]|%b %d, %Y}</b><br>"
        "Total Gas Used: %{x:,.0f}<br>"
        "Transactions: %{y:,.0f}"
        "<extra></extra>"
    )
)

# Trendline color
fig.data[-1].update(
    line=dict(color="orange", width=1)
)

fig.show()

## 4. Gas Price vs Transactions

In [ ]:
fig = px.scatter(
    analysis_df,
    x="avg_gas_price_gwei",
    y="transaction_count",
    title="Gas Price vs Transactions (r = 0.065)",
    labels={
        "avg_gas_price_gwei": "Average Gas Price (Gwei)",
        "transaction_count": "Transaction Count"
    },
    trendline="ols",
    custom_data=["date"]
)

fig.update_layout(
    template="plotly_dark",
    hovermode="closest",
    hoverlabel=dict(
        bgcolor="rgba(30, 30, 30, 0.80)",
        font_size=13
    ),
    height=600,
    margin=dict(l=70, r=40, t=100, b=80),
    xaxis=dict(
        title="Average Gas Price (Gwei)",
        tickformat=".2f",
        fixedrange=False
    ),
    yaxis=dict(
        title="Transaction Count",
        tickformat=",",
        fixedrange=True
    )
)

fig.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "<b>%{customdata[0]|%b %d, %Y}</b><br>"
        "Gas Price: %{x:.2f} Gwei<br>"
        "Transactions: %{y:,.0f}"
        "<extra></extra>"
    )
)

# Trendline color
fig.data[-1].update(
    line=dict(color="orange", width=1)
)

fig.show()

## 5. Gas Utilization vs Transactions

In [ ]:
fig = px.scatter(
    analysis_df,
    x="avg_gas_utilization_pct",
    y="transaction_count",
    title="Gas Utilization vs Transactions (r = -0.115)",
    labels={
        "avg_gas_utilization_pct": "Average Gas Utilization (%)",
        "transaction_count": "Transaction Count"
    },
    trendline="ols",
    custom_data=["date"]
)

fig.update_layout(
    template="plotly_dark",
    hovermode="closest",
    hoverlabel=dict(
        bgcolor="rgba(30, 30, 30, 0.80)",
        font_size=13
    ),
    height=600,
    margin=dict(l=70, r=40, t=100, b=80),
    xaxis=dict(
        title="Average Gas Utilization (%)",
        tickformat=".2f",
        ticksuffix="%",
        fixedrange=False
    ),
    yaxis=dict(
        title="Transaction Count",
        tickformat=",",
        fixedrange=True
    )
)

fig.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "<b>%{customdata[0]|%b %d, %Y}</b><br>"
        "Gas Utilization: %{x:.2f}%<br>"
        "Transactions: %{y:,.0f}"
        "<extra></extra>"
    )
)

# Trendline color
fig.data[-1].update(
    line=dict(color="orange", width=1)
)

fig.show()

## 8. Trend Comparison

We compare the standardized movement of the major activity metrics so that variables with different units can be viewed on the same scale.


In [ ]:
fig = px.line(
    standardized,
    x="date",
    y=metrics,
    title="Standardized Ethereum Network Activity Metrics",
    labels={
        "date": "Date",
        "value": "Standardized Value (z-score)",
        "variable": "Metric"
    }
)

fig.update_layout(
    template="plotly_dark",
    hovermode="x unified",
    hoverlabel=dict(
        bgcolor="rgba(30, 30, 30, 0.80)",
        font_size=13
    ),
    height=600,
    margin=dict(l=70, r=40, t=110, b=100),
    xaxis=dict(
        title="Date",
        hoverformat="%b %d, %Y",
        tickformat="%b %d",
        nticks=15,
        tickangle=-45,
        fixedrange=False
    ),
    yaxis=dict(
        title="Standardized Value (z-score)",
        fixedrange=True,
        zeroline=True,
        zerolinewidth=1
    ),
    legend=dict(
        title="Metric",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    )
)

fig.update_traces(
    mode="lines+markers",
    marker=dict(size=5)
)

fig.for_each_trace(
    lambda trace: trace.update(
        name={
            "transaction_count": "Transaction Count",
            "active_addresses": "Active Addresses",
            "total_gas_used": "Total Gas Used",
            "avg_gas_price_gwei": "Average Gas Price",
            "avg_gas_utilization_pct": "Gas Utilization"
        }.get(trace.name, trace.name),
        hovertemplate=(
            "<b>%{fullData.name}</b>: %{y:.2f}"
            "<extra></extra>"
        )
    )
)

fig.show()

## 9. Finding → Evidence → Meaning

The following cell generates the evidence-based summary directly from the calculated correlations. This avoids manually entering correlation values and reduces the risk of reporting stale results.


In [137]:
strongest = relationship_results.iloc[0]
weakest = relationship_results.iloc[-1]

def strength_label(r):
    a = abs(r)
    if a >= 0.80:
        return "strong"
    elif a >= 0.50:
        return "moderate"
    elif a >= 0.30:
        return "weak-to-moderate"
    else:
        return "weak"

print("### Finding")
print(
    f"The strongest linear association among the five specified relationships "
    f"was **{strongest['relationship']}**, with a Pearson correlation of "
    f"**{strongest['correlation']:.3f}**."
)

print("\n### Evidence")
for _, row in relationship_results.iterrows():
    print(
        f"- **{row['relationship']}:** "
        f"Pearson correlation = **{row['correlation']:.3f}** "
        f"({strength_label(row['correlation'])} linear association)."
    )

print(
    f"- The analysis covers **29 complete days**, from "
    f"**{start_date.date()} to {end_date.date()}**."
)
print(
    "- August 24 and September 23 were excluded because they are partial "
    "boundary days."
)

print("\n### Meaning")
print(
    "The correlations describe how the daily metrics moved together during "
    "the analysis period. A positive correlation indicates that higher values "
    "of one metric tended to occur alongside higher values of the other; a "
    "negative correlation indicates the opposite pattern."
)
print(
    "These relationships describe statistical association only. They do not "
    "establish that changes in one metric caused changes in another."
)


### Finding
The strongest linear association among the five specified relationships was **Transactions vs active addresses**, with a Pearson correlation of **0.874**.

### Evidence
- **Transactions vs active addresses:** Pearson correlation = **0.874** (strong linear association).
- **Active addresses vs transactions per active address:** Pearson correlation = **-0.839** (strong linear association).
- **Transactions vs gas used:** Pearson correlation = **-0.223** (weak linear association).
- **Gas utilization vs transaction activity:** Pearson correlation = **-0.115** (weak linear association).
- **Gas price vs network activity:** Pearson correlation = **0.065** (weak linear association).
- The analysis covers **29 complete days**, from **2026-08-24 to 2026-09-23**.
- August 24 and September 23 were excluded because they are partial boundary days.

### Meaning
The correlations describe how the daily metrics moved together during the analysis period. A positive correlation indicates tha

## Limitations

- The correlation analysis is based on **29 complete daily observations**, so the results describe relationships within this specific analysis period and should not be assumed to represent longer-term Ethereum network behavior.

- **Pearson correlation measures linear association.** A weak correlation does not necessarily mean that no relationship exists; a nonlinear relationship may still be present.

- The relationship between **active addresses and transactions per active address** should be interpreted cautiously because transactions per active address is calculated using active addresses as its denominator. This creates a mathematical dependency between the two variables, so the observed correlation should not be interpreted as an independent behavioral relationship.

- The analysis uses **daily aggregated metrics**, which can hide relationships or variation that may occur at a smaller time scale, such as hourly or transaction-level activity.

- The standardized trend analysis places metrics with different units on a common z-score scale for comparison, but it does not measure the strength of relationships between those metrics.

- The observed correlations describe **statistical associations and do not establish causation**. Additional analysis would be required to determine whether external events or changes in one network metric contributed to changes in another.